In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the Kickstarter dataset

df = pd.read_csv("../data/raw/kickstarter_raw.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (378661, 15)


In [3]:
# Display the first 5 rows

df.head()

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,usd_pledged_real,usd_goal_real
0,1000002330,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09,1000.0,2015-08-11 12:12:28,0.0,failed,0,GB,0.0,0.0,1533.95
1,1000003930,Greeting From Earth: ZGAC Arts Capsule For ET,Narrative Film,Film & Video,USD,2017-11-01,30000.0,2017-09-02 04:43:57,2421.0,failed,15,US,100.0,2421.0,30000.00
2,1000004038,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26,45000.0,2013-01-12 00:20:50,220.0,failed,3,US,220.0,220.0,45000.00
3,1000007540,ToshiCapital Rekordz Needs Help to Complete Album,Music,Music,USD,2012-04-16,5000.0,2012-03-17 03:24:11,1.0,failed,1,US,1.0,1.0,5000.00
4,1000011046,Community Film Project: The Art of Neighborhoo...,Film & Video,Film & Video,USD,2015-08-29,19500.0,2015-07-04 08:35:03,1283.0,canceled,14,US,1283.0,1283.0,19500.00


In [4]:
# Check column names and data types

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Column names:
['ID', 'name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'pledged', 'state', 'backers', 'country', 'usd pledged', 'usd_pledged_real', 'usd_goal_real']

Data types:
ID                    int64
name                    str
category                str
main_category           str
currency                str
deadline                str
goal                float64
launched                str
pledged             float64
state                   str
backers               int64
country                 str
usd pledged         float64
usd_pledged_real    float64
usd_goal_real       float64
dtype: object


In [5]:
# Check missing values

missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
ID                     0
name                   4
category               0
main_category          0
currency               0
deadline               0
goal                   0
launched               0
pledged                0
state                  0
backers                0
country                0
usd pledged         3797
usd_pledged_real       0
usd_goal_real          0
dtype: int64


In [6]:
# Check for duplicate rows

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [7]:
# Check the distribution of project states

print("Project state distribution:")
print(df["state"].value_counts())

Project state distribution:
state
failed        197719
successful    133956
canceled       38779
undefined       3562
live            2799
suspended       1846
Name: count, dtype: int64


In [8]:
# Keep only completed projects: successful and failed

df = df[df["state"].isin(["successful", "failed"])].copy()

# Create binary target
df["target"] = df["state"].map({
    "successful": 1,
    "failed": 0
})

print("Dataset after keeping successful and failed projects:")
print("Shape:", df.shape)

print("\nTarget distribution:")
print(df["target"].value_counts())

Dataset after keeping successful and failed projects:
Shape: (331675, 16)

Target distribution:
target
0    197719
1    133956
Name: count, dtype: int64


In [9]:
# Remove rows with missing project names

missing_name_count = df["name"].isnull().sum()

print("Missing project names before cleaning:", missing_name_count)

df = df.dropna(subset=["name"]).copy()

print("Missing project names after cleaning:", df["name"].isnull().sum())
print("New dataset shape:", df.shape)

Missing project names before cleaning: 3
Missing project names after cleaning: 0
New dataset shape: (331672, 16)


In [10]:
# Convert date columns to datetime

df["deadline"] = pd.to_datetime(df["deadline"])
df["launched"] = pd.to_datetime(df["launched"])

print("Date columns converted successfully!")

print("\nData types after conversion:")
print(df[["deadline", "launched"]].dtypes)

Date columns converted successfully!

Data types after conversion:
deadline    datetime64[us]
launched    datetime64[us]
dtype: object


In [11]:
# Create campaign duration in days

df["duration_days"] = (
    df["deadline"] - df["launched"]
).dt.total_seconds() / (24 * 60 * 60)

print("Duration feature created successfully!")

print("\nDuration statistics:")
print(df["duration_days"].describe())

Duration feature created successfully!

Duration statistics:
count    331672.000000
mean         33.389368
std          12.723333
min           0.005058
25%          29.088935
50%          29.656765
75%          35.705767
max          91.962650
Name: duration_days, dtype: float64


In [12]:
# Check for invalid campaign durations

invalid_duration = (df["duration_days"] <= 0).sum()

print("Number of invalid durations:", invalid_duration)

print("\nMinimum duration:", df["duration_days"].min())
print("Maximum duration:", df["duration_days"].max())

Number of invalid durations: 0

Minimum duration: 0.0050578703703703706
Maximum duration: 91.96265046296297


In [13]:
# Remove identifier and data-leakage columns

columns_to_remove = [
    "ID",
    "pledged",
    "backers",
    "usd_pledged",
    "usd_pledged_real"
]

df = df.drop(columns=columns_to_remove)

print("Removed columns:")
print(columns_to_remove)

print("\nRemaining columns:")
print(df.columns.tolist())

print("\nCurrent dataset shape:", df.shape)

KeyError: "['usd_pledged'] not found in axis"

In [14]:
# Check the current column names

print(df.columns.tolist())

['ID', 'name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'pledged', 'state', 'backers', 'country', 'usd pledged', 'usd_pledged_real', 'usd_goal_real', 'target', 'duration_days']


In [15]:
# Check which leakage columns are present

leakage_columns = [
    "pledged",
    "backers",
    "usd_pledged",
    "usd_pledged_real"
]

print("Leakage columns found in dataset:")

for column in leakage_columns:
    print(column, "→", column in df.columns)

Leakage columns found in dataset:
pledged → True
backers → True
usd_pledged → False
usd_pledged_real → True


In [16]:
# Remove identifier and data-leakage columns

columns_to_remove = [
    "ID",
    "pledged",
    "backers",
    "usd_pledged_real"
]

df = df.drop(columns=columns_to_remove)

print("Removed columns:")
print(columns_to_remove)

print("\nRemaining columns:")
print(df.columns.tolist())

print("\nCurrent dataset shape:", df.shape)

Removed columns:
['ID', 'pledged', 'backers', 'usd_pledged_real']

Remaining columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'state', 'country', 'usd pledged', 'usd_goal_real', 'target', 'duration_days']

Current dataset shape: (331672, 13)


In [17]:
# Final missing-value check after cleaning

print("Missing values in remaining columns:")
print(df.isnull().sum())


Missing values in remaining columns:
name               0
category           0
main_category      0
currency           0
deadline           0
goal               0
launched           0
state              0
country            0
usd pledged      210
usd_goal_real      0
target             0
duration_days      0
dtype: int64


In [18]:
# Remove remaining data-leakage column

df = df.drop(columns=["usd_pledged"])

print("usd_pledged removed successfully!")

print("\nRemaining columns:")
print(df.columns.tolist())

print("\nFinal shape:")
print(df.shape)

KeyError: "['usd_pledged'] not found in axis"

In [19]:
# Check the current dataset after cleaning

print("Current columns:")
print(df.columns.tolist())

print("\nCurrent dataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

Current columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'state', 'country', 'usd pledged', 'usd_goal_real', 'target', 'duration_days']

Current dataset shape:
(331672, 13)

Missing values:
name               0
category           0
main_category      0
currency           0
deadline           0
goal               0
launched           0
state              0
country            0
usd pledged      210
usd_goal_real      0
target             0
duration_days      0
dtype: int64


In [20]:
# Safely remove the remaining data-leakage column

df = df.drop(columns=["usd_pledged"], errors="ignore")

print("usd_pledged removed successfully!")

print("\nRemaining columns:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

usd_pledged removed successfully!

Remaining columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'state', 'country', 'usd pledged', 'usd_goal_real', 'target', 'duration_days']

Dataset shape:
(331672, 13)

Missing values:
name               0
category           0
main_category      0
currency           0
deadline           0
goal               0
launched           0
state              0
country            0
usd pledged      210
usd_goal_real      0
target             0
duration_days      0
dtype: int64


In [21]:
# Remove the remaining data-leakage column

df = df.drop(columns=["usd pledged"])

print("Remaining columns:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

Remaining columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'state', 'country', 'usd_goal_real', 'target', 'duration_days']

Dataset shape:
(331672, 12)

Missing values:
name             0
category         0
main_category    0
currency         0
deadline         0
goal             0
launched         0
state            0
country          0
usd_goal_real    0
target           0
duration_days    0
dtype: int64


In [22]:
# Remove the original state column
# The target column already represents success/failure

df = df.drop(columns=["state"])

print("State column removed successfully!")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nFinal dataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

State column removed successfully!

Final columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'country', 'usd_goal_real', 'target', 'duration_days']

Final dataset shape:
(331672, 11)

Missing values:
name             0
category         0
main_category    0
currency         0
deadline         0
goal             0
launched         0
country          0
usd_goal_real    0
target           0
duration_days    0
dtype: int64


In [23]:
# Save the cleaned dataset

output_path = "../data/processed/kickstarter_cleaned.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Saved to:", output_path)

Cleaned dataset saved successfully!
Saved to: ../data/processed/kickstarter_cleaned.csv
